Load the data, import libraries, show the sample

In [5]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
url = "https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz"
players = pd.read_csv(url)
players.sample(10)

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
127,Amateur,True,56bae029998e9fef1cf18f614dc0a5b300640494787c10...,0.0,Peyton,Male,21,NaN,NaN
11,Pro,True,4caa42e1b20511552434978171dcf7283fb6eb857eb871...,0.0,Daniela,Male,17,NaN,NaN
110,Amateur,True,827f9a82b47cdf21098bc7bb1d4241550dc6146ba9ae8b...,0.0,Sean,Male,22,NaN,NaN
124,Beginner,True,ed3e6d043dee74fad932be44d0600f4502055198c51192...,0.0,Hamish,Male,17,NaN,NaN
18,Amateur,True,ab1f44f93c3b828f55458971db393052d9711df3e0e7ff...,0.5,Marley,Male,17,NaN,NaN
165,Regular,True,c121e4d197469bea90e21c0495001f4e21824adb98cbc6...,0.1,Rupert,Male,21,NaN,NaN
44,Veteran,True,8d2eed1f399e0d77cebb8fcc48ed19ad2fa8e3bb3fa683...,2.2,Cyrus,Male,24,NaN,NaN
28,Amateur,True,4b01bce3f141289709e8278b02ba5d2aaa7105d7ccb9c7...,1.8,Luca,Male,23,NaN,NaN
171,Beginner,False,80afc8e7137de6a232421e926c1e6e64ddeef1d8157c44...,1.8,Amelia,Male,32,NaN,NaN
136,Regular,True,657e5c2da9dad8ec67b8f2875d98c290dc97573d86b770...,0.0,Jamal,Male,20,NaN,NaN


Extract the data to 3 colunms

In [6]:
player_selected = players[["subscribe","played_hours","age"]]
player_selected

,subscribe,played_hours,age
0,True,30.3,9
1,True,3.8,17
2,False,0.0,17
3,True,0.7,21
4,True,0.1,21
...,...,...,...
191,True,0.0,17
192,False,0.3,22
193,False,0.0,17
194,False,2.3,17


Set the random seed

In [7]:
np.random.seed(1)

Split the train set and test set

In [12]:
player_train, player_test = train_test_split(
    player_selected, train_size=0.75, stratify=player_selected["subscribe"]
)
player_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 147 entries, 51 to 151
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   subscribe     147 non-null    bool   
 1   played_hours  147 non-null    float64
 2   age           147 non-null    int64  
dtypes: bool(1), float64(1), int64(1)
memory usage: 3.6 KB


standarize the data

In [14]:
player_preprocessor = make_column_transformer(
    (StandardScaler(), ["played_hours", "age"]),
)

train the classifier with the k = 3

In [16]:
X = player_train[["played_hours", "age"]]
y = player_train["subscribe"]
knn = KNeighborsClassifier(n_neighbors=3)
knn_pipeline = make_pipeline(player_preprocessor, knn)
knn_pipeline.fit(X, y)

knn_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['played_hours', 'age'])])),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=3))])

predict the test set with the model and test the score

In [20]:
player_test["predicted"] = knn_pipeline.predict(player_test[["played_hours", "age"]])
player_test[["subscribe", "predicted"]]
knn_pipeline.score(
    player_test[["played_hours", "age"]],
    player_test["subscribe"]
)

0.7142857142857143

Cross validation

In [22]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()
player_tune_pipe = make_pipeline(player_preprocessor, knn)

parameter_grid = {
    "kneighborsclassifier__n_neighbors": range(2, 20, 1),
}

player_tune_grid = GridSearchCV(
    estimator=player_tune_pipe,
    param_grid=parameter_grid,
    cv=10
)

player_tune_grid.fit(
    player_train[["played_hours", "age"]],
    player_train["subscribe"]
)

accuracies_grid = pd.DataFrame(player_tune_grid.cv_results_)
accuracies_grid.info()

X = training_data
y = training_labels

knn_grid.fit(X, y)

cv_results = pd.DataFrame(knn_grid.cv_results_)

cross_val_plot = (
    alt.Chart(cv_results)
    .mark_line(point=True)
    .encode(
        x=alt.X("param_n_neighbors", title="Number of Neighbors (k)", scale=alt.Scale(zero=False)),
        y=alt.Y("mean_test_score", title="Mean Cross-Validation Accuracy", scale=alt.Scale(zero=False))
    )
)

cross_val_plot

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 19 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   mean_fit_time                            18 non-null     float64
 1   std_fit_time                             18 non-null     float64
 2   mean_score_time                          18 non-null     float64
 3   std_score_time                           18 non-null     float64
 4   param_kneighborsclassifier__n_neighbors  18 non-null     int64  
 5   params                                   18 non-null     object 
 6   split0_test_score                        18 non-null     float64
 7   split1_test_score                        18 non-null     float64
 8   split2_test_score                        18 non-null     float64
 9   split3_test_score                        18 non-null     float64
 10  split4_test_score                        18 non-null